In [12]:
import pandas as pd
import numpy as np
import datasets

DATA = '~/data/spancat/'
dataset_path = DATA+'strategies-ds.hf'

In [2]:
df = pd.read_csv(DATA+'strategies-df.csv').iloc[1:, :30][['OE_response', 'Rewrite',
       'Highlight', 'Summarize', 'Quizzing', 'Examples', 'Comparisons',
       'Explain', 'MA', 'AllExplain', 'ALLComparison']]
col_list = list(df.columns)
col_list[0] = 'text'
df.columns = col_list
df = df.dropna(how='all').fillna(0).replace(' ', 0)
for i in df.columns[1:]:
    df[i] = df[i].astype(float)
labels = [np.array(label[1:]) for label in df.values.tolist()]
df['labels'] = labels
df['text']= df['text'].astype(str)

df_examples = df[df['Examples']==1].reset_index()

In [3]:
from transformers import pipeline

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
pipe = pipeline("text-generation", model_name, device='cuda')

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [14]:
def load_dataset(dataset_path):
    ds = datasets.load_from_disk(dataset_path)
    return ds

ds = load_dataset(dataset_path)
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 1212
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 304
    })
})

In [15]:
def generate_content(example):
    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>
    You are a student describing the strategies you used to study for a test. 
    Please rephrase the following text without changing the meaning or adding any new information.
    Do not write more words than the original.:
    {example}
    <|start_header_id|>student<|end_header_id|>
    """
    output = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    generated_text = output[0]['generated_text'].split('<|start_header_id|>student<|end_header_id|>\n')[1]
    return generated_text

In [ ]:

ds['train']['synthetic_text'] = [generate_content(x) for x in ds['train']['text']]
# df_examples['synthetic_Examples'] = df_examples['text'].progress_apply(generate_content)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [67]:
i = 6

print(df_examples['text'].iloc[i])

print('----')
print(df_examples['synthetic_Examples'].iloc[i])

Most often, I thoroughly review notes to the point where i can apply concepts. 
----
     When it comes to studying for a test, I typically start by thoroughly reviewing my notes. This involves going through each page, reading every sentence, and making sure I understand the material. I make sure to highlight important points, underline key terms, and take notes in the margins to help me retain the information better. By doing this, I'm able to internalize the concepts and make connections between different ideas. For example, I might summarize a complex concept in my own words, or create a diagram to illustrate a process. As I review my notes, I also try to apply the concepts to real-life scenarios or hypothetical situations. This helps me to see the relevance of the material and how it can be used in practical ways. By the time I'm done reviewing my notes, I feel confident that I can apply the concepts and understand the material at a deeper level.
